### PCA
- 简介
- 推导过程
    - 前置背景
    - PCA的桥梁
    - 代入SVD
    

- 应用场景
    - 降维
    - 图片压缩

- sklearn应用

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as pe
import pandas as pd
import seaborn as sns
import matplotlib as mpl
%matplotlib inline
data=pe.data.iris()
columns_name=data.columns

### 推导过程
#### 前置背景

[特征值分解和奇异值分解](../Math/linear/eigenvalue_svd.ipynb)

#### PCA的桥梁
1. 协方差矩阵
有了解了特征值、特征向量以及 SVD后。这些知识看起来和 PCA 没有什么直接关系，但实际上 PCA 的核心恰恰就是
通过协方差矩阵的特征值和特征向量，寻找数据变化最大的方向。在介绍PCA之前先来看看协方差矩阵
从数据矩阵开始
假设现在有n个样本，有m个特征将数据排列成矩阵
$$
X = \begin{bmatrix} 
x_{11} & x_{12} & \cdots & x_{1m}  \\
x_{21} & x_{22} & \cdots & x_{2m}  \\
\vdots & \vdots &       & \vdots \\
x_{n1} & x_{n2} & \cdots & x_{nm}  \\
\end{bmatrix}
$$
- 每一行代表一个样本
- 每一列代表一个特征
例如：
$$
X = \begin{bmatrix} 
身高 & 体重  \\
173 & 78 \\
184 & 88 \\
\end{bmatrix}
$$
每个样本看成空间中的一个点。因为PCA关注的数据围绕均值变化的情况，因此需要减去每个特征自己的均值，比如j列特征的均值为$x_j=\frac{1}{n}\sum_{i=1}^n x_{ij}$，然后对每个特征进行处理$X_c=X-X^{～}$ ($X^{～}$表示每列均值矩阵)，就得到X去中心化矩阵，此时这个矩阵描述的就不是身高和体重本身多大，而是每个样本相对应平均值偏了多少。
然后让中心化矩阵内积，构成协方差矩阵
$$
C=\frac{1}{n-1}X_c^TX_c
$$
这里C是一个mxm矩阵它实际计算不同特征之间的内积

$$
C_{ij}=\frac{1}{n-1}\sum_{k=1}^n(x_{ki}-x_i^～)(x_{kj}-x_j^～)
$$
- 当i=j，对应的矩阵的对角线，它的值为j特征的数据的方差。记$C_{ii}=Var(X_i)$
- 当i!=j，对应的矩阵上三角或下三角（由于他是对称矩阵，所以他们两个是相等的），它的值特征i与特征j数据的协方差。记$C_{ij}=Cov(X_{i},X_{j})$
因此协方差矩阵可以理解为，用一个矩阵同时记录特征自身的变化程度，以及不同特征之间的变化关系。
> 这个矩阵表示数据之间的关联程度，假如上三角的某个$C_{ij}$很大，说明两个特征具有很强的相关。这里可以用余弦定理说明，协方差矩阵有去中心化的X^TX构造，当i!=j时，$C_{ij}=\frac{1}{n-1}Cov(X_{i},X_{j})=X_{i}^TX_j$ 协方差矩阵里的ij项写成$X_{i}$与$X_j$
的内积，$Cov(X_{i},X_{j})=\frac{1}{n-1}X_{i}^TX_j=|X_i||X_j|cos(\theta)$，此时我们在原有的矩阵除以$\sigma_i$和$\sigma_j$它们是特征ij数据的标准差$\sigma_i=\frac{||X_{i}||}{\sqrt{n-1}}$（标准差看作去中心化矩阵特征列的长度），所以相除后通过内积展开化简只剩下一个cos，这时候原来协方差矩阵就变成，对角线为1(皮尔逊相关系数矩阵)，其他项为ij特征向量之间的余弦值。因此就可以看到值接近于1关联度就越高

> 如果 \(C_{ij}\) 的绝对值较大，说明两个特征具有较强的共同变化程度，但协方差本身会受到特征量纲和尺度影响，因此不能仅根据数值大小直接比较不同特征之间的相关程度。标准化后得到的相关系数则消除了这种尺度影响。
$$
X = \begin{bmatrix} 
1 & \cos(\theta_{12})  \\
\cos(\theta_{21}) & 1  \\
\end{bmatrix}
$$


2.  特征值与PCA成分关系

假设在二维平面上$X_c \in R^{nx2}，w \in R^2$，现在我们想换一个新的方向来观察数据，放弃原来的x,y。设这个新方向为w，并且它的长度为1(||w||=1)，我们把所有数据投影到这个方向$z=X_cw$
- w 随机的一个方向，并不知道具体指
- $X_cw$是投影后的新的坐标

现在引入一个新问题，在那个方向上投影能使数据变化最大（也就是投影后的方差最大），数据做了中心化处理它的均值为0，那么它的方差$Var(z)= \frac{1}{n-1}z^Tz$。对这个式子进行改写向量形式
$$
\frac{1}{n-1}z^Tz=\frac{1}{n-1} (X_cw)^T(X_cw)=\frac{1}{n-1} w^TX_c^TX_cw
$$
可以发现中间的那串式子就是上面构建的协方差矩阵，对其化简
$$
Var(z)=\frac{1}{n-1} w^TCw
$$
问题就变成 $\max w^TCw$。
现在接着来看，C矩阵它是一个对称矩阵可以正交对角化 $C=Q\Lambda Q^T$ （$Cq_i=\lambda_i q_i$）
$$
w^TCw=w^TQ\Lambda Q^Tw
$$
令$y=Q^Tw$，由于||w||=1，Q又是正交矩阵||y||=1，
$$
w^TCw=y^T\Lambda y = \lambda_1 y_1^2 + \lambda_2 y_2^2 ...
$$
所以 $\max w^TCw = \lambda_1$ 是C矩阵特征最大的特征值。取最大值的w方向为$w=q_1$，是$\frac{1}{n-1}X^TX$矩阵的最大特征值所对应的特征值

$w^TX_c^TX_cw=\lambda_1 q_1^Tq_1$。也就是说协方差矩阵C最大特征值对应的特征向量为是第一主成分（表示）数据投影到这个方向以后能获得最大的方差，紧接着找后面特征值小的对应的向量。
- $q_1$第一主成分方向
- $z_1=X_cq_1$ 第一主成分投影后的坐标 - 新坐标 
$$\lambda_1>\lambda_2>\lambda_3>..\lambda_i$$
由于$\frac{1}{n-1}X_c^TX_c$是对称矩阵，满足对角化有一组相互垂直基底$q_1,q_2,..,q_i$。所以PCA本质上就是利用协方差矩阵的特征向量，重新构造一个适合描述数据的正交坐标系

3. PCA 实际意义
PCA 可以看成一次坐标系变换：放弃原来的特征坐标轴，转而使用数据自身的主要变化方向作为新的坐标轴。第一主轴上的数据方差最大，第二主轴在与第一主轴正交的条件下方差最大，以此类推。并且这组基底$q_1,q_2,..q_n$都是相互垂直


4. PCA 与 SVD 的关系
前面已经推导出

$$
C=Q\Lambda Q^T
$$
Q中的列向量维PCA主城分。而SVD

$$
X_c=U \Sigma V^T
$$
那他们之间怎么联系的了。假设对数据矩阵X做了中心化处理
 $(U^TU=I)$
$$
X_c^TX_c=V \Sigma^T U^T U \Sigma V^T=V \Sigma^T  \Sigma   V^T  
$$
因此

$$
C=Q\Lambda Q^T =  V \Sigma^T  \Sigma V^T
$$
得到下列关系($\Sigma^T  \Sigma$也是斜对角矩阵)
$$
\lambda_i=\frac{\sigma_i^2}{n-1}
$$
所以也可以得到Q=V

所以PCA的主方向，就是中心化数据矩阵做SVD后得到的右奇异向量V



5. 数据降维
PCA找出m个新坐标基底，从中选出前k列主轴作为新的坐标系来表示数据从而实现将数据从m->k维。比如3x3特征向量矩阵（表示3个特征）

$$
Q = \begin{bmatrix} 
&  & \\
q_1 & q_2 & q_3  \\
&  & \\
\end{bmatrix}
$$，
X为 mx3 (m个数据，3个特征)，
$$
X = \begin{bmatrix} 
x_{11} & x_{12} & x_{13}  \\
x_{21} & x_{22} & x_{23}  \\
\vdots & \vdots & \vdots \\
x_{m1} & x_{m2} & x_{m3}  \\
\end{bmatrix}
$$

现在减少特征维度，把$q_3$剔除(3x2)，直接通过$XQ$这样就变成mx2的数据矩阵(m个数据，2个特征)从而完成降维。在这里可能会想为什么能丢掉了，可以通过保留信息比来看$方差比=\frac{\sum_{i=1}^k \lambda_i}{\sum_{i=1}^m\lambda_i}$，例如$\lambda=[50,30,15,5]$保留前3个，方差比为95%意味着用3个主成分表示原来四维数据95%的总方差

6. PCA过程
先对数据中心化$X_c=X-X^～$，在计算协方差矩阵 $C=\frac{1}{n-1}X_c^TX_c$对协方差矩阵进行特征值分解
$C=Q\Lambda Q^T$，得到 $\lambda_1>\lambda_2>\lambda_3>..\lambda_i$,对应的特征向量q，数据变化最大的几个方向，依此类推，如果保留前k方向，那么原数据将投影到这个k个方向得到降为后数据$Z=X_cQ_k$



